## Task 3.1: Prepare data and train/test split

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split, GridSearchCV,
    cross_val_score, cross_val_predict
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

df = pd.read_csv("/content/appointments_clean.csv")

features = [
    "age", "distance_km", "fee_inr",
    "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]

X = df[features].copy()
y = df["no_show"]

X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1, "No": 0
})

X = pd.get_dummies(
    X, columns=["booking_channel", "visit_type"]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Features:", X.shape[1])
print("Training:", len(X_train))
print("Testing:", len(X_test))
print("Test no-shows:", y_test.sum())

baseline = (y_test == 0).mean()
print("Baseline accuracy:", round(baseline, 4))


Features: 11
Training: 792
Testing: 198
Test no-shows: 49
Baseline accuracy: 0.7525


## Task 3.2: Logistic Regression

In [2]:

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

lr_pred = lr.predict(X_test_s)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred, zero_division=0))
print("Recall:", recall_score(y_test, lr_pred, zero_division=0))
print("F1:", f1_score(y_test, lr_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))
print("Confusion matrix:")
print(confusion_matrix(y_test, lr_pred))

coef = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_[0]
})
print(coef.to_string(index=False))

reminder_idx = X.columns.get_loc("reminder_sent")
reminder_coef = lr.coef_[0][reminder_idx]
reminder_scale = scaler.scale_[reminder_idx]

print("Odds ratio per 1 SD:", np.exp(reminder_coef))
print("Odds ratio No to Yes:",
      np.exp(reminder_coef / reminder_scale))


Accuracy: 0.7878787878787878
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579
Confusion matrix:
[[139  10]
 [ 32  17]]
                feature  coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658
Odds ratio per 1 SD: 0.559524141750829
Odds ratio No to Yes: 0.3093980903124433


## Task 3.3: Decision Tree

In [3]:

tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

tree_small = DecisionTreeClassifier(
    max_depth=3, random_state=42
)
tree_small.fit(X_train, y_train)

for name, model in [
    ("Full Tree", tree_full),
    ("Depth-3 Tree", tree_small)
]:
    print(name)
    print("Train accuracy:",
          accuracy_score(y_train, model.predict(X_train)))
    print("Test accuracy:",
          accuracy_score(y_test, model.predict(X_test)))
    print()


Full Tree
Train accuracy: 1.0
Test accuracy: 0.6717171717171717

Depth-3 Tree
Train accuracy: 0.821969696969697
Test accuracy: 0.7777777777777778



The unrestricted tree shows overfitting because its training accuracy is much higher than its test accuracy. Limiting tree depth reduces this gap.

## Task 3.4: Random Forest and model comparison

In [4]:

rf = RandomForestClassifier(
    n_estimators=100, random_state=42
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("RF Recall:", recall_score(y_test, rf_pred))
print("RF ROC-AUC:", roc_auc_score(y_test, rf_prob))

importance = pd.Series(
    rf.feature_importances_, index=X.columns
).sort_values(ascending=False)

print("\nTop 3 features:")
print(importance.head(3))

models = {
    "Logistic Regression": (lr_pred, lr_prob),
    "Depth-3 Tree": (
        tree_small.predict(X_test),
        tree_small.predict_proba(X_test)[:, 1]
    ),
    "Random Forest": (rf_pred, rf_prob)
}

comparison = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred), 3),
        "Recall": round(recall_score(y_test, pred), 3),
        "ROC-AUC": round(roc_auc_score(y_test, prob), 3)
    }
    for name, (pred, prob) in models.items()
])

print(comparison.to_string(index=False))


RF Accuracy: 0.7474747474747475
RF Recall: 0.30612244897959184
RF ROC-AUC: 0.7057252431173813

Top 3 features:
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64
              Model  Accuracy  Recall  ROC-AUC
Logistic Regression     0.788   0.347    0.789
       Depth-3 Tree     0.778   0.224    0.700
      Random Forest     0.747   0.306    0.706


## Task 3.5: Hyperparameter tuning and model selection

In [5]:

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)


Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673242251739405


In [6]:

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

candidates = {
    "Logistic Regression": lr_pipe,
    "Depth-3 Tree": DecisionTreeClassifier(
        max_depth=3, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    )
}

cv_scores = {
    name: cross_val_score(
        model, X_train, y_train,
        cv=5, scoring="roc_auc"
    ).mean()
    for name, model in candidates.items()
}

cv_scores["Tuned Random Forest"] = grid.best_score_

print(pd.Series(cv_scores).sort_values(ascending=False))

winner_name = max(cv_scores, key=cv_scores.get)

if winner_name == "Tuned Random Forest":
    winner = grid.best_estimator_
else:
    winner = candidates[winner_name]

print("Selected model:", winner_name)


Logistic Regression    0.773714
Tuned Random Forest    0.767324
Random Forest          0.746602
Depth-3 Tree           0.678449
dtype: float64
Selected model: Logistic Regression


In [7]:

oof_probs = cross_val_predict(
    winner,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

threshold_results = []

for threshold in [0.5, 0.35]:
    pred = (oof_probs >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, pred, zero_division=0
        ),
        "recall": recall_score(
            y_train, pred, zero_division=0
        ),
        "f1": f1_score(
            y_train, pred, zero_division=0
        ),
        "flagged": pred.sum()
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df)

best_threshold = threshold_df.loc[
    threshold_df["f1"].idxmax(), "threshold"
]

print("Chosen threshold:", best_threshold)


   threshold  precision    recall        f1  flagged
0       0.50   0.575342  0.248521  0.347107       73
1       0.35   0.512987  0.467456  0.489164      154
Chosen threshold: 0.35


## Final test evaluation and CSV export

In [8]:

winner.fit(X_train, y_train)

test_probs = winner.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= best_threshold).astype(int)

print("Final model:", winner_name)
print("Threshold:", best_threshold)
print("Precision:",
      precision_score(y_test, test_flags, zero_division=0))
print("Recall:",
      recall_score(y_test, test_flags, zero_division=0))
print("Flagged appointments:", test_flags.sum())
print("ROC-AUC:", roc_auc_score(y_test, test_probs))

risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags
})

risk_scores.to_csv("test_risk_scores.csv", index=False)

print("Saved test_risk_scores.csv")
print("Shape:", risk_scores.shape)
print(risk_scores.head())


Final model: Logistic Regression
Threshold: 0.35
Precision: 0.5714285714285714
Recall: 0.5714285714285714
Flagged appointments: 49
ROC-AUC: 0.788933022873579
Saved test_risk_scores.csv
Shape: (198, 4)
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
